# 🔍 FakeScope — Fake Engagement Detection in Social Media
**Behavioural Analytics Hackathon | Problem Statement 3**

| Item | Detail |
|---|---|
| **Dataset** | airt-ml/twitter-human-bots (HuggingFace, CC-BY-SA 3.0) |
| **Model** | Random Forest Classifier (scikit-learn) |
| **Expected Outputs** | Authenticity Score · Bot Probability · Behavioural Anomaly Explanation |

▶️ Run cells **top to bottom**. Everything is fully automated.


## 📦 Step 1 — Install Dependencies

In [ ]:
!pip install pyarrow pandas scikit-learn matplotlib joblib -q
print('✅ All packages ready')

## 📂 Step 2 — Download Real Dataset from HuggingFace

In [ ]:
import pandas as pd
import numpy as np
import os, warnings, json
warnings.filterwarnings('ignore')

os.makedirs('data', exist_ok=True)
os.makedirs('outputs', exist_ok=True)
os.makedirs('models', exist_ok=True)

print('⬇️  Downloading dataset from HuggingFace...')
URL = 'https://huggingface.co/datasets/airt-ml/twitter-human-bots/resolve/refs%2Fconvert%2Fparquet/default/train/0000.parquet'
df_raw = pd.read_parquet(URL)
df_raw.to_parquet('data/twitter_human_bots.parquet', index=False)

print(f'✅ Downloaded! Shape: {df_raw.shape}')
print(f'Columns: {list(df_raw.columns)}')
print(f'\nLabel distribution:')
print(df_raw['account_type'].value_counts())
df_raw.head(3)

## 🔧 Step 3 — Feature Engineering
We engineer **30 behavioural features** covering ALL 4 indicators from the problem statement:
- ⏱️ **Timing Regularity** — posting pace consistency signals
- 💥 **Engagement Burst Patterns** — abnormal activity spikes
- 🌐 **Network Interaction Patterns** — follower/following anomalies
- 🗣️ **Linguistic Consistency** — username, bio, and text patterns

In [ ]:
df = df_raw.copy()
df['label'] = (df['account_type'] == 'bot').astype(int)

# ════════════════════════════════════════════════════════════
# GROUP 1 — ACTIVITY & TIMING REGULARITY
# Bots post at superhuman rates with robotic consistency
# ════════════════════════════════════════════════════════════
df['tweets_per_day']    = df['average_tweets_per_day'].fillna(0).clip(0, 1000)
df['account_age_days']  = df['account_age_days'].fillna(0).clip(0, 6000)
df['total_tweets']      = df['statuses_count'].fillna(0).clip(0, 3_000_000)
df['total_likes_given'] = df['favourites_count'].fillna(0).clip(0, 1_000_000)

# Tweet density = total tweets / age — high = bot
df['tweet_density']     = (df['total_tweets'] / (df['account_age_days'] + 1)).clip(0, 500)
df['likes_per_day']     = (df['total_likes_given'] / (df['account_age_days'] + 1)).clip(0, 1000)

# TIMING REGULARITY PROXY:
# Bots tweet uniformly — their tweets_per_day is stable and extreme.
# We model regularity using the ratio of tweets to account age buckets.
# High tweet rate with very young account = robotic regularity signal.
df['timing_regularity_score'] = (
    df['tweets_per_day'] / (np.log1p(df['account_age_days']) + 1)
).clip(0, 500)

# Normalised productivity: how much of life has been spent tweeting
df['lifetime_tweet_rate'] = (df['total_tweets'] / (df['account_age_days'] * 24 + 1)).clip(0, 100)

# ════════════════════════════════════════════════════════════
# GROUP 2 — ENGAGEMENT BURST PATTERNS
# Sudden spikes: new account + huge followers, abnormal rate
# ════════════════════════════════════════════════════════════
df['is_high_tweeter']     = (df['tweets_per_day'] > 50).astype(int)
df['abnormal_tweet_rate'] = (df['tweets_per_day'] > 100).astype(int)

# Burst index: young account with disproportionate tweet volume
df['burst_index'] = (
    df['total_tweets'] / (df['account_age_days'] + 1)
).clip(0, 1000)
df['burst_flag'] = (df['burst_index'] > 20).astype(int)

# Sudden fame: new account already has lots of followers (bought)
df['sudden_fame'] = (
    (df['account_age_days'] < 30) & (df['followers_count'] > 1000)
).astype(int)

# Engagement paradox: many followers but zero likes given
df['engagement_paradox'] = (
    (df['followers_count'] > 500) & (df['total_likes_given'] == 0)
).astype(int)

# ════════════════════════════════════════════════════════════
# GROUP 3 — NETWORK INTERACTION PATTERNS
# Bots follow many, gain few real followers back
# ════════════════════════════════════════════════════════════
df['followers_count']       = df['followers_count'].fillna(0).clip(0, 50_000_000)
df['friends_count']         = df['friends_count'].fillna(0).clip(0, 5_000_000)
df['follower_friend_ratio'] = (df['followers_count'] / (df['friends_count'] + 1)).clip(0, 500)
df['friends_to_followers']  = (df['friends_count'] / (df['followers_count'] + 1)).clip(0, 500)
df['likes_to_tweets']       = (df['total_likes_given'] / (df['total_tweets'] + 1)).clip(0, 500)

# Network anomaly: extreme imbalance between follow and follower counts
df['network_anomaly_flag'] = (
    (df['friends_count'] > 2000) & (df['follower_friend_ratio'] < 0.1)
).astype(int)

# ════════════════════════════════════════════════════════════
# GROUP 4 — LINGUISTIC CONSISTENCY
# Bots use templated bios, numeric usernames, no location
# ════════════════════════════════════════════════════════════
# Profile completeness signals
df['has_description']    = df['description'].notna().astype(int)
df['description_length'] = df['description'].fillna('').str.len().clip(0, 280)
df['description_word_count'] = df['description'].fillna('').str.split().str.len().clip(0, 60)
df['has_location']       = df['location'].notna().astype(int)
df['has_default_pic']    = df['default_profile_image'].astype(int)
df['is_default_profile'] = df['default_profile'].astype(int)
df['is_verified']        = df['verified'].astype(int)
df['has_geo']            = df['geo_enabled'].astype(int)

# Screen name linguistic signals (bots often have numeric/random names)
df['screen_name_length']      = df['screen_name'].fillna('').str.len().clip(0, 50)
df['screen_name_digit_ratio'] = (
    df['screen_name'].fillna('').apply(lambda x: sum(c.isdigit() for c in x) / (len(x) + 1))
).clip(0, 1)   # High digit ratio = bot (e.g. user_48291847)

# Bio linguistic richness
df['bio_has_url'] = df['description'].fillna('').str.contains('http|www', case=False).astype(int)
df['bio_is_generic'] = (
    (df['description_length'] > 0) & (df['description_length'] < 15)
).astype(int)   # Very short bio = likely templated

# Profile completeness score (0-5)
df['profile_completeness'] = (
    df['has_description'] +
    df['has_location'] +
    (1 - df['has_default_pic']) +
    (1 - df['is_default_profile']) +
    df['has_geo']
)

# ════════════════════════════════════════════════════════════
# FINAL FEATURE LIST — 30 Features covering all 4 PS indicators
# ════════════════════════════════════════════════════════════
FEATURES = [
    # Timing Regularity
    'tweets_per_day', 'account_age_days', 'tweet_density',
    'timing_regularity_score', 'lifetime_tweet_rate',
    # Engagement Burst Patterns
    'total_tweets', 'total_likes_given', 'likes_per_day',
    'is_high_tweeter', 'abnormal_tweet_rate',
    'burst_index', 'burst_flag', 'sudden_fame', 'engagement_paradox',
    # Network Interaction Patterns
    'followers_count', 'friends_count',
    'follower_friend_ratio', 'friends_to_followers',
    'likes_to_tweets', 'network_anomaly_flag',
    # Linguistic Consistency
    'has_description', 'description_length', 'description_word_count',
    'has_location', 'has_default_pic', 'is_default_profile',
    'is_verified', 'has_geo', 'profile_completeness',
    'screen_name_length', 'screen_name_digit_ratio',
    'bio_has_url', 'bio_is_generic'
]

df_clean = df[FEATURES + ['label']].copy()
df_clean.replace([np.inf, -np.inf], np.nan, inplace=True)
df_clean.fillna(0, inplace=True)
df_clean['label'] = df_clean['label'].astype(int)
df_clean.to_csv('data/processed_dataset.csv', index=False)

print('✅ Feature engineering complete!')
print(f'   Records  : {len(df_clean):,}')
print(f'   Features : {len(FEATURES)}')
print(f'   Bots     : {df_clean["label"].sum():,}  ({df_clean["label"].mean()*100:.1f}%)')
print(f'   Humans   : {(df_clean["label"]==0).sum():,}  ({(1-df_clean["label"].mean())*100:.1f}%)')
print(f'\n   Feature groups:')
print(f'   ⏱️  Timing Regularity      : tweets_per_day, timing_regularity_score, lifetime_tweet_rate ...')
print(f'   💥 Engagement Burst        : burst_index, burst_flag, sudden_fame, engagement_paradox ...')
print(f'   🌐 Network Interaction     : follower_friend_ratio, friends_to_followers, network_anomaly_flag ...')
print(f'   🗣️  Linguistic Consistency  : screen_name_digit_ratio, bio_is_generic, description_word_count ...')
df_clean.describe().round(2)

## 🤖 Step 4 — Train Random Forest Model

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix, roc_curve,
                              classification_report)
from sklearn.utils.class_weight import compute_class_weight
import joblib

X = df_clean[FEATURES]
y = df_clean['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')

# Handle class imbalance with balanced weights
cw = compute_class_weight('balanced', classes=np.array([0,1]), y=y_train)
cw_dict = {0: cw[0], 1: cw[1]}
print(f'Class weights → Human: {cw[0]:.2f}  Bot: {cw[1]:.2f}')

print('\n🌲 Training Random Forest (300 trees)...')
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight=cw_dict,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)
print('✅ Training complete!')

## 📊 Step 5 — Evaluate Model

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

metrics = {
    'accuracy'  : round(accuracy_score(y_test,  y_pred), 4),
    'precision' : round(precision_score(y_test, y_pred), 4),
    'recall'    : round(recall_score(y_test,    y_pred), 4),
    'f1_score'  : round(f1_score(y_test,        y_pred), 4),
    'roc_auc'   : round(roc_auc_score(y_test,   y_prob), 4),
}

cv     = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_auc = cross_val_score(model, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)
metrics['cv_auc_mean'] = round(cv_auc.mean(), 4)
metrics['cv_auc_std']  = round(cv_auc.std(),  4)

print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print('   MODEL EVALUATION RESULTS')
print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
for k, v in metrics.items():
    print(f'   {k:<16}: {v}')
print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Human', 'Bot']))

with open('outputs/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print('✅ Metrics saved → outputs/metrics.json')

## 📊 Step 6 — Generate All Visualizations
Covers: Confusion Matrix · ROC Curve · Feature Importance · Distributions · Feature Group Summary

In [ ]:
import matplotlib.pyplot as plt

DARK_BG='#0D0F14'; CARD_BG='#1A1D26'; BORDER='#2A2D3A'
GREEN='#4ECCA3'; RED='#FF4757'; YELLOW='#FFE66D'; GREY='#8B8FA8'

def dark_fig(w=7, h=5):
    fig, ax = plt.subplots(figsize=(w,h), facecolor=DARK_BG)
    ax.set_facecolor(CARD_BG)
    for sp in ax.spines.values(): sp.set_color(BORDER)
    ax.tick_params(colors=GREY)
    ax.xaxis.label.set_color(GREY)
    ax.yaxis.label.set_color(GREY)
    ax.title.set_color('white')
    return fig, ax

# ── Plot 1: Confusion Matrix ──────────────────────────────
cm = confusion_matrix(y_test, y_pred)
fig, ax = dark_fig(6, 5)
ax.imshow(cm, cmap='Blues', alpha=0.85)
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(['Human','Bot'], fontsize=13, color='white')
ax.set_yticklabels(['Human','Bot'], fontsize=13, color='white')
ax.set_xlabel('Predicted', fontsize=13); ax.set_ylabel('Actual', fontsize=13)
ax.set_title('Confusion Matrix', fontsize=15, fontweight='bold', pad=14)
for i in range(2):
    for j in range(2):
        c = 'white' if cm[i,j]>cm.max()/2 else GREY
        ax.text(j, i, f'{cm[i,j]:,}', ha='center', va='center', fontsize=18, color=c, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Confusion matrix saved')

In [ ]:
# ── Plot 2: ROC Curve ─────────────────────────────────────
fpr, tpr, _ = roc_curve(y_test, y_prob)
fig, ax = dark_fig(7, 5)
ax.plot(fpr, tpr, color=RED, lw=2.5, label=f'ROC AUC = {metrics["roc_auc"]:.4f}')
ax.fill_between(fpr, tpr, alpha=0.08, color=RED)
ax.plot([0,1],[0,1],'--',lw=1,color=BORDER)
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve — Bot Detection', fontsize=14, fontweight='bold')
ax.legend(fontsize=12, framealpha=0.2, facecolor=CARD_BG, labelcolor='white')
ax.grid(alpha=0.15, color=BORDER)
plt.tight_layout()
plt.savefig('outputs/roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ ROC curve saved')

In [ ]:
# ── Plot 3: Feature Importance (Top 20) ───────────────────
feat_imp = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=True)
top20    = feat_imp.tail(20)

# Color-code by feature group
GROUP_COLORS = {
    'tweets_per_day':'#FF4757','account_age_days':'#FF4757','tweet_density':'#FF4757',
    'timing_regularity_score':'#FF4757','lifetime_tweet_rate':'#FF4757',
    'total_tweets':'#FFE66D','total_likes_given':'#FFE66D','likes_per_day':'#FFE66D',
    'is_high_tweeter':'#FFE66D','abnormal_tweet_rate':'#FFE66D',
    'burst_index':'#FFE66D','burst_flag':'#FFE66D','sudden_fame':'#FFE66D','engagement_paradox':'#FFE66D',
    'followers_count':'#4ECCA3','friends_count':'#4ECCA3',
    'follower_friend_ratio':'#4ECCA3','friends_to_followers':'#4ECCA3',
    'likes_to_tweets':'#4ECCA3','network_anomaly_flag':'#4ECCA3',
    'has_description':'#A78BFA','description_length':'#A78BFA','description_word_count':'#A78BFA',
    'has_location':'#A78BFA','has_default_pic':'#A78BFA','is_default_profile':'#A78BFA',
    'is_verified':'#A78BFA','has_geo':'#A78BFA','profile_completeness':'#A78BFA',
    'screen_name_length':'#A78BFA','screen_name_digit_ratio':'#A78BFA',
    'bio_has_url':'#A78BFA','bio_is_generic':'#A78BFA'
}
bar_colors = [GROUP_COLORS.get(f, GREY) for f in top20.index]

fig, ax = dark_fig(11, 9)
bars = ax.barh(top20.index, top20.values, color=bar_colors, height=0.7, edgecolor=BORDER, linewidth=0.5)
ax.set_xlabel('Feature Importance (Gini)', fontsize=12)
ax.set_title('Top 20 Behavioural Features — Bot vs Human\n'
             '🔴 Timing  🟡 Burst  🟢 Network  🟣 Linguistic', fontsize=13, fontweight='bold')
ax.grid(axis='x', alpha=0.15, color=BORDER)
for bar, val in zip(bars, top20.values):
    ax.text(bar.get_width()+0.001, bar.get_y()+bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9, color=GREY)
plt.tight_layout()
plt.savefig('outputs/feature_importance.png', dpi=150, bbox_inches='tight')
plt.savefig('outputs/shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Feature importance saved')

In [ ]:
# ── Plot 4: Feature Group Importance (PS-aligned) ─────────
# Shows importance contribution per PS behavioural indicator group
timing_feats  = ['tweets_per_day','account_age_days','tweet_density','timing_regularity_score','lifetime_tweet_rate']
burst_feats   = ['total_tweets','total_likes_given','likes_per_day','is_high_tweeter','abnormal_tweet_rate','burst_index','burst_flag','sudden_fame','engagement_paradox']
network_feats = ['followers_count','friends_count','follower_friend_ratio','friends_to_followers','likes_to_tweets','network_anomaly_flag']
ling_feats    = ['has_description','description_length','description_word_count','has_location','has_default_pic','is_default_profile','is_verified','has_geo','profile_completeness','screen_name_length','screen_name_digit_ratio','bio_has_url','bio_is_generic']

group_imp = {
    '⏱️ Timing\nRegularity'   : feat_imp[timing_feats].sum(),
    '💥 Engagement\nBurst'    : feat_imp[burst_feats].sum(),
    '🌐 Network\nInteraction' : feat_imp[network_feats].sum(),
    '🗣️ Linguistic\nConsistency': feat_imp[ling_feats].sum(),
}

fig, axes = plt.subplots(1, 2, figsize=(13, 5), facecolor=DARK_BG)

# Bar chart
ax = axes[0]; ax.set_facecolor(CARD_BG)
for sp in ax.spines.values(): sp.set_color(BORDER)
gcolors = [RED, YELLOW, GREEN, '#A78BFA']
bars = ax.bar(list(group_imp.keys()), list(group_imp.values()), color=gcolors, edgecolor=BORDER, width=0.5)
ax.set_ylabel('Total Feature Importance', fontsize=11, color=GREY)
ax.set_title('Importance by Behavioural\nIndicator Group (PS-Aligned)', fontsize=12, color='white', fontweight='bold')
ax.tick_params(colors=GREY, labelsize=9)
for bar, val in zip(bars, group_imp.values()):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10, color='white', fontweight='bold')
ax.grid(axis='y', alpha=0.15, color=BORDER)

# Pie chart
ax2 = axes[1]; ax2.set_facecolor(CARD_BG)
wedge_props = {'edgecolor': DARK_BG, 'linewidth': 2}
ax2.pie(list(group_imp.values()), labels=list(group_imp.keys()),
        colors=gcolors, autopct='%1.1f%%', startangle=90,
        textprops={'color':'white','fontsize':10}, wedgeprops=wedge_props)
ax2.set_title('Feature Group Distribution', fontsize=12, color='white', fontweight='bold')

plt.tight_layout()
plt.savefig('outputs/group_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Group importance chart saved')

In [ ]:
# ── Plot 5: Feature Distributions: Bot vs Human ───────────
key_feats = [
    'timing_regularity_score', 'burst_index',
    'follower_friend_ratio',   'screen_name_digit_ratio',
    'profile_completeness',    'engagement_paradox'
]
feat_labels = [
    '⏱️ Timing Regularity Score', '💥 Burst Index',
    '🌐 Follower/Friend Ratio',   '🗣️ Screen Name Digit Ratio',
    '👤 Profile Completeness',    '💔 Engagement Paradox'
]

fig, axes = plt.subplots(2, 3, figsize=(14, 8), facecolor=DARK_BG)
fig.suptitle('Key Behavioural Features: Bot vs Human Distribution',
             fontsize=14, color='white', fontweight='bold', y=1.01)

df_full = pd.concat([X, y], axis=1)
for ax, feat, label in zip(axes.flat, key_feats, feat_labels):
    ax.set_facecolor(CARD_BG)
    for sp in ax.spines.values(): sp.set_color(BORDER)
    ax.hist(df_full[df_full['label']==0][feat], bins=40, alpha=0.7, color=GREEN, density=True, label='Human')
    ax.hist(df_full[df_full['label']==1][feat], bins=40, alpha=0.7, color=RED,   density=True, label='Bot')
    ax.set_title(label, fontsize=10, color='white')
    ax.tick_params(colors=GREY, labelsize=8)
    ax.legend(fontsize=8, framealpha=0.2, facecolor=CARD_BG, labelcolor='white')
    ax.grid(alpha=0.1, color=BORDER)

plt.tight_layout()
plt.savefig('outputs/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Distribution plots saved')

## 🎯 Step 7 — Generate Expected Outputs (PS Requirements)
The problem statement requires 3 specific outputs:
1. **Authenticity Score** (organic-ness of account)
2. **Bot Probability** (likelihood of being a bot)
3. **Behavioural Anomaly Explanation** (which signals triggered the flag)


In [ ]:
def explain_account(row_dict, model, features, feat_imp_series):
    """
    Generates all 3 required PS outputs for a given account:
    - Bot Probability
    - Authenticity Score
    - Behavioural Anomaly Explanation
    """
    X_in = pd.DataFrame([row_dict])[features]
    prob_bot   = model.predict_proba(X_in)[0][1]
    bot_score  = round(prob_bot * 100, 1)
    auth_score = round(100 - bot_score, 1)

    # Verdict
    if bot_score >= 70:   verdict = 'BOT DETECTED'
    elif bot_score >= 40: verdict = 'SUSPICIOUS'
    else:                 verdict = 'ORGANIC USER'

    # Behavioural Anomaly Explanation — rule-based flags
    anomalies = []
    v = row_dict
    if v.get('timing_regularity_score', 0) > 10:
        anomalies.append('[TIMING] Robotic posting regularity — inhuman tweet rate relative to account age')
    if v.get('abnormal_tweet_rate', 0):
        anomalies.append('[TIMING] Abnormal tweet rate — posting >100 tweets/day')
    if v.get('burst_flag', 0):
        anomalies.append('[BURST] Burst activity detected — disproportionate tweet volume for account age')
    if v.get('sudden_fame', 0):
        anomalies.append('[BURST] Sudden fame — new account (<30 days) with >1000 followers')
    if v.get('engagement_paradox', 0):
        anomalies.append('[BURST] Engagement paradox — many followers but zero likes given')
    if v.get('network_anomaly_flag', 0):
        anomalies.append('[NETWORK] Network anomaly — follows >2000 accounts but tiny follower ratio')
    if v.get('friends_to_followers', 0) > 5:
        anomalies.append('[NETWORK] Extreme follow imbalance — following far more than followers')
    if v.get('screen_name_digit_ratio', 0) > 0.4:
        anomalies.append('[LINGUISTIC] Numeric username — high digit ratio, typical of generated bot names')
    if v.get('has_default_pic', 0):
        anomalies.append('[LINGUISTIC] Default profile picture — profile never personalised')
    if v.get('bio_is_generic', 0):
        anomalies.append('[LINGUISTIC] Generic/empty bio — likely templated or unfilled')
    if v.get('profile_completeness', 5) < 2:
        anomalies.append('[LINGUISTIC] Low profile completeness — multiple setup signals missing')

    # Top contributing features for this account
    top_features = feat_imp_series.sort_values(ascending=False).head(5).index.tolist()

    return {
        'bot_probability'   : bot_score,
        'authenticity_score': auth_score,
        'verdict'           : verdict,
        'anomaly_explanation': anomalies if anomalies else ['No significant anomalies detected'],
        'top_driving_features': top_features
    }

# ── Demo: Run on 5 accounts from test set ─────────────────
sample_indices = X_test.sample(5, random_state=7).index
print('='*60)
print('  EXPECTED OUTPUT DEMO — 5 Sample Accounts')
print('='*60)
for idx in sample_indices:
    row  = X_test.loc[idx].to_dict()
    true = 'BOT' if y_test.loc[idx] == 1 else 'HUMAN'
    result = explain_account(row, model, FEATURES, feat_imp)
    print(f'\n  Account #{idx}  [TRUE LABEL: {true}]')
    print(f'  ├─ Bot Probability    : {result["bot_probability"]}%')
    print(f'  ├─ Authenticity Score : {result["authenticity_score"]}%')
    print(f'  ├─ Verdict            : {result["verdict"]}')
    print(f'  ├─ Behavioural Anomalies:')
    for a in result['anomaly_explanation']:
        print(f'  │    • {a}')
    print(f'  └─ Top Driving Features: {result["top_driving_features"]}')
    print()

# Save the explain function and output format to disk for dashboard use
sample_output = explain_account(X_test.iloc[0].to_dict(), model, FEATURES, feat_imp)
with open('outputs/sample_output_format.json', 'w') as f:
    json.dump(sample_output, f, indent=2)
print('\n✅ Sample output format saved → outputs/sample_output_format.json')

## 💾 Step 8 — Save Model & All Artifacts

In [ ]:
import joblib
joblib.dump(model, 'models/rf_model.pkl')
pd.Series(FEATURES).to_csv('models/features.csv', index=False, header=False)
feat_imp.to_csv('models/feature_importance.csv', header=['importance'])

print('✅ Model saved              → models/rf_model.pkl')
print('✅ Feature list saved       → models/features.csv')
print('✅ Feature importance saved → models/feature_importance.csv')

print('\n' + '='*50)
print('  🎉 FULL PIPELINE COMPLETE')
print('='*50)
print(f'  Accuracy  : {metrics["accuracy"]}')
print(f'  Precision : {metrics["precision"]}')
print(f'  Recall    : {metrics["recall"]}')
print(f'  F1 Score  : {metrics["f1_score"]}')
print(f'  ROC AUC   : {metrics["roc_auc"]}')
print(f'  CV AUC    : {metrics["cv_auc_mean"]} ± {metrics["cv_auc_std"]}')
print('='*50)
print('\n  PS3 Outputs Generated:')
print('  ✅ Authenticity Score   (100 - Bot Probability)')
print('  ✅ Bot Probability      (model.predict_proba)')
print('  ✅ Behavioural Anomaly Explanation (explain_account)')
print('\n  All outputs in: outputs/')
for f in os.listdir('outputs'):
    size = os.path.getsize(f'outputs/{f}')
    print(f'    {f}  ({size/1024:.1f} KB)')

## 📥 Step 9 — Download Everything as ZIP

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('fakescope_outputs', 'zip', '.', 'outputs')
shutil.make_archive('fakescope_models',  'zip', '.', 'models')
shutil.make_archive('fakescope_data',    'zip', '.', 'data')

files.download('fakescope_outputs.zip')
files.download('fakescope_models.zip')
files.download('fakescope_data.zip')

print('\n✅ Downloaded! Unzip into your project folder, then run:')
print('   streamlit run app.py')

## 🧪 Bonus — Manual Account Test
Verify all 3 PS outputs on handcrafted bot and human examples.

In [ ]:
bot_account = {
    'tweets_per_day':150, 'account_age_days':10, 'total_tweets':1500,
    'total_likes_given':0, 'tweet_density':150, 'likes_per_day':0,
    'timing_regularity_score':150/(np.log1p(10)+1), 'lifetime_tweet_rate':1500/(10*24+1),
    'is_high_tweeter':1, 'abnormal_tweet_rate':1,
    'burst_index':150, 'burst_flag':1, 'sudden_fame':1, 'engagement_paradox':1,
    'followers_count':8000, 'friends_count':12000,
    'follower_friend_ratio':0.67, 'friends_to_followers':1.5,
    'likes_to_tweets':0, 'network_anomaly_flag':1,
    'has_description':0, 'description_length':0, 'description_word_count':0,
    'has_location':0, 'has_default_pic':1, 'is_default_profile':1,
    'is_verified':0, 'has_geo':0, 'profile_completeness':0,
    'screen_name_length':15, 'screen_name_digit_ratio':0.6,
    'bio_has_url':0, 'bio_is_generic':0
}

human_account = {
    'tweets_per_day':3, 'account_age_days':1500, 'total_tweets':4500,
    'total_likes_given':12000, 'tweet_density':3, 'likes_per_day':8,
    'timing_regularity_score':3/(np.log1p(1500)+1), 'lifetime_tweet_rate':4500/(1500*24+1),
    'is_high_tweeter':0, 'abnormal_tweet_rate':0,
    'burst_index':3, 'burst_flag':0, 'sudden_fame':0, 'engagement_paradox':0,
    'followers_count':920, 'friends_count':650,
    'follower_friend_ratio':1.42, 'friends_to_followers':0.71,
    'likes_to_tweets':2.67, 'network_anomaly_flag':0,
    'has_description':1, 'description_length':110, 'description_word_count':18,
    'has_location':1, 'has_default_pic':0, 'is_default_profile':0,
    'is_verified':0, 'has_geo':1, 'profile_completeness':5,
    'screen_name_length':9, 'screen_name_digit_ratio':0.0,
    'bio_has_url':0, 'bio_is_generic':0
}

print('='*60)
print('  MANUAL TEST — All 3 PS Expected Outputs')
print('='*60)
for name, account in [('🤖 BOT ACCOUNT', bot_account), ('👤 HUMAN ACCOUNT', human_account)]:
    result = explain_account(account, model, FEATURES, feat_imp)
    print(f'\n{name}')
    print(f'  OUTPUT 1 — Bot Probability    : {result["bot_probability"]}%')
    print(f'  OUTPUT 2 — Authenticity Score : {result["authenticity_score"]}%')
    print(f'  OUTPUT 3 — Verdict            : {result["verdict"]}')
    print(f'  Behavioural Anomaly Explanation:')
    for a in result['anomaly_explanation']:
        print(f'    • {a}')
    print(f'  Top Driving Features: {result["top_driving_features"]}')